In [1]:
import cv2
import numpy as np
from keras.models import load_model

# Load face detector
classifier = cv2.CascadeClassifier(
    r"C:\Users\patra\OneDrive\Desktop\FACE DETECTION\haarcascade_frontalface_default (1).xml"
)

# Load new trained model
model = load_model(
    r"C:\Users\patra\OneDrive\Desktop\FACE DETECTION\final_model.h5"
)

# New trained people
labels = ['Sapna', 'Simran', 'dibiya']


def preprocess(img):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, (100, 100))
    img = cv2.equalizeHist(img)
    img = img.reshape(1, 100, 100, 1)
    img = img / 255.0
    return img


# Open laptop camera
camera = cv2.VideoCapture(0)

while True:

    ret, frame = camera.read()

    if not ret:
        print("Could not access laptop camera")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = classifier.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(50, 50)
    )

    for x, y, w, h in faces:

        face = frame[y:y+h, x:x+w]

        processed_face = preprocess(face)

        prediction = model.predict(
            processed_face,
            verbose=0
        )[0]

        predicted_class = np.argmax(prediction)

        name = labels[predicted_class]

        confidence = float(prediction[predicted_class]) * 100

        # Draw face rectangle
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (255, 0, 0),
            2
        )

        # Display name and confidence
        text = f"{name} - {confidence:.1f}%"

        cv2.putText(
            frame,
            text,
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 0, 0),
            2
        )

    cv2.imshow("Face Recognition", frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


camera.release()
cv2.destroyAllWindows()